In [0]:
#Step 1st: Importing required librarieslike functions, deltatable, datetime, uuid.

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
#Catalog usage initiation.

spark.sql("Use catalog data_novacart_databricks")

In [0]:
#Schema creation for silver_schema.

spark.sql("CREATE SCHEMA IF NOT EXISTS silver_schema")

silver_run_id = str(uuid.uuid4())
print("Current silver run id",silver_run_id)

In [0]:
#Step 2nd: Creation of Silver control table that can hold water mark for each source table.
#It will help pipeline remember : latest timestamp(ts) already processed, latest primarykey(pk) processed at that timestamp,how many rows were written in the latest run. This will make it incremental and re-run safe.

spark.sql(""" 
        
create table if not exists data_novacart_databricks.silver_schema.processing_control(
    layer string,
    entity_name string,
    last_processed_bronze_run_id string,
    last_processed_bronze_ingested_at timestamp,
    rows_merged bigint,
    run_status string,
    silver_run_id string,
    updated_at timestamp
    
)   
using delta
"""

)

In [0]:
#step 3rd: Creation of Helper Functions.
#this cell will contain reusable logic for silver and will have following helper functions:
#1. upsert_to_silver() - it will merge clean/transformed rows into silver target table.
#2. get_last_processed_bronze_ingested_at() - to get latest silver watermark.
#3. upsert_silver_control() - Updates the Silver Control table.
#4. get_incremental_bronze() - Read only new Bronze rows that silver has not processed yet.

def upsert_to_silver(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (
            dt.alias("target")
            .merge(df_source.alias("source"), f"target.{join_key} = source.{join_key}")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
#get_last_processed_bronze_ingested_at() - to get latest silver watermark.

def get_last_processed_bronze_ingested_at(table_name:str):
    ctrl = (spark.table("data_novacart_databricks.silver_schema.processing_control")
                .filter(
                    (F.col("layer")=="Silver")&
                    (F.col("entity_name")==table_name)&
                    (F.col("run_status")=="SUCCESS")
                    
                )
                .orderBy(F.col("updated_at").desc())
                .limit(1)
    )    
    rows= ctrl.collect()
    if not rows:
        return None
   
    return[0]["last_processed_bronze_ingested_at"]


In [0]:
#3. upsert_silver_control() - Updates the Silver Control table.

def upsert_silver_control(entity_name, last_processed_bronze_run_id, last_processed_bronze_ingested_at, rows_merged):
    ctrl_df = spark.createDataFrame(
        [
            (
                "Silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                int(rows_merged),
                "SUCCESS",
                silver_run_id,
                datetime.now(),
            )
        ],
        schema="""
        layer string,
        entity_name string,
        last_processed_bronze_run_id string,
        last_processed_bronze_ingested_at timestamp,
        rows_merged bigint,
        run_status string,
        silver_run_id string,
        updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "data_novacart_databricks.silver_schema.processing_control")
    (
        dt.alias("t")
        .merge(ctrl_df.alias("s"), "t.layer = s.layer AND t.entity_name = s.entity_name")
        .whenMatchedUpdate(set={"last_processed_bronze_run_id": "s.last_processed_bronze_run_id", "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at", "rows_merged": "s.rows_merged", "run_status": "s.run_status", "silver_run_id": "s.silver_run_id", "updated_at": "s.updated_at"})
        .whenNotMatchedInsertAll()
        .execute()
    )


In [0]:
#4. get_incremental_bronze() - Read only new Bronze rows that silver has not processed yet.

def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df, last_ingested_at
    
    return bronze_df.filter(F.col("bronze_ingested_at") > F.lit(last_ingested_at)), last_ingested_at